# Week 03: Retry & Resilience — ScraperFlow V3

**Goal:** Handle transient failures (timeouts, 5xx, connection resets) with one consistent, tested policy instead of ad hoc `try/except` per scraper. One `@retry` decorator, one exception hierarchy, zero duplicated retry logic.

**ScraperFlow deliverable:** `@retry` decorator with exponential backoff + jitter, `TransientFetchError`/`PermanentFetchError` exception hierarchy.

**Disciplines this week:** Python, Architecture, Testing

---
## Overview

### What are we building in ScraperFlow this week?

V3 adds resilience to the fetch layer. Running V2 against real sites surfaces genuine, repeated network failures — timeouts, HTTP 5xx responses, connection resets. Instead of scattering `try/except` blocks with ad hoc retry loops throughout the codebase, V3 introduces:

1. **An exception hierarchy** that classifies failures by type: `TransientFetchError` (worth retrying) vs. `PermanentFetchError` (don't bother)
2. **A `@retry` decorator** that catches narrowly (only `TransientFetchError`), backs off exponentially with jitter, and gives up loudly after a bounded number of attempts

### Which version does this map to?

**Version 3 — Retry & Resilience** (Chapter 6 of the handbook)

### What concepts do we need to learn before implementing?

- **Decorators and closures** — the mechanism that makes wrapping behavior around functions possible
- **Custom exception hierarchies** — how to encode failure classification into the type system
- **Cross-cutting concerns** — the architectural concept that tells us retry belongs in a decorator, not inline
- **Exponential backoff with jitter** — the specific retry strategy that avoids hammering struggling servers

### Impact on future versions

This exception hierarchy becomes the **shared vocabulary** every later version classifies failure by:
- V6 extends it with `StorageError`
- V7's Worker uses these types to decide whether a URL gets requeued (transient) or marked permanently failed (permanent)
- V10's distributed system adds a second resilience layer (requeue with cooldown) on top of V3's immediate retry

---
## Concepts by Discipline

### Python Concepts

**Core (full depth):** Decorators & closures — the load-bearing mechanism this week. You need to understand how closures capture variables and how decorators wrap control flow to build the `@retry` decorator correctly.

**Core (full depth):** Custom exception hierarchies — the classification system that tells the decorator what to catch.

**Supporting (lighter):** `functools.wraps` — one-line metadata preservation. Usage-level knowledge is sufficient.

### Architecture Concepts

**Core (full depth):** Cross-cutting concerns — the architectural reasoning that justifies putting retry in a decorator rather than inline.

**Core (full depth):** Exponential backoff with jitter — the specific algorithm for spacing retries.

**Supporting (lighter):** Transient vs. permanent failure classification — a design heuristic, not a deep algorithm.

### Testing Concepts

**Supporting (lighter):** Monkeypatching `time.sleep` — a testing technique to keep the suite fast.

---
## CORE: Decorators & Closures

### 1. Terminology

| Term | Definition |
|---|---|
| **Closure** | A function object that remembers values from its enclosing lexical scope, even after the outer function has returned. The "closed-over" variables are stored in the function's `__closure__` attribute. |
| **Free variable** | A variable used inside a function that is neither a local variable nor a parameter of that function — it's defined in an enclosing scope. |
| **Decorator** | A callable that takes a function as input and returns a new function (or the same function, modified). Applied with `@decorator` syntax above a function definition. |
| **Decorator factory** | A function that *returns* a decorator. Used when the decorator needs parameters: `@retry(max_attempts=3)` — `retry(max_attempts=3)` is the factory call, and its return value is the actual decorator. |
| **Wrapper function** | The inner function returned by a decorator that "wraps" the original function, typically calling it somewhere inside while adding behavior before/after/around. |
| **`functools.wraps`** | A decorator (yes, a decorator for decorators) that copies metadata (`__name__`, `__doc__`, `__module__`, `__qualname__`, `__dict__`, `__wrapped__`) from the original function to the wrapper. |

### 2. What, Why & Consequences

**What is it?**

A **closure** is a function that captures variables from its enclosing scope. A **decorator** is a specific application of closures: a function that wraps another function to inject behavior (logging, retry, timing, caching) without modifying the wrapped function's source code.

**Why does it exist?**

The fundamental problem: you have behavior that needs to be applied to *many* functions but isn't part of those functions' core responsibility. Without decorators, you'd:
- Copy-paste the behavior into every function (violates DRY)
- Use inheritance and override methods (too heavyweight for behavior injection)
- Manually wrap each function at the call site (clutters calling code)

Decorators solve this by letting you declare "this function should have retry logic" in one place (`@retry`) without the function's body knowing about it.

**What goes wrong without it?**

Without decorators, retry logic for ScraperFlow would look like:
```python
# In every function that does network I/O:
def fetch_page(url):
    for attempt in range(3):
        try:
            response = requests.get(url)
            return response.text
        except requests.Timeout:
            time.sleep(2 ** attempt)
    raise FetchError("gave up")

def fetch_sitemap(url):
    for attempt in range(3):  # Same loop, duplicated
        try:
            response = requests.get(url)
            return response.text
        except requests.Timeout:
            time.sleep(2 ** attempt)
    raise FetchError("gave up")
```

The retry logic is repeated, untestable in isolation, and impossible to change globally (need to update every copy). With a decorator:
```python
@retry(max_attempts=3)
def fetch_page(url):
    response = requests.get(url)
    return response.text  # Clean — just the core responsibility
```

### 3. How It Works (Internals)

#### Closures — The Foundation

A closure is created whenever an inner function references a variable from its enclosing function:

```
def outer(x):           ← outer defines variable x
    def inner(y):       ← inner uses x (a free variable)
        return x + y   ← x is "closed over" — captured by inner
    return inner        ← outer returns the inner function object

add_five = outer(5)     ← outer(5) runs, returns inner with x=5 captured
add_five(3)             → 8 (x=5 is still accessible even though outer returned)
```

**Where does `x` live after `outer` returns?**

Python stores free variables in **cell objects** attached to the function's `__closure__` tuple. The cell holds a reference to the value, keeping it alive even after the enclosing scope's stack frame is gone.

```
add_five.__closure__           → (<cell at 0x...>,)
add_five.__closure__[0].cell_contents  → 5
add_five.__code__.co_freevars  → ('x',)
```

#### Decorators — Execution Flow

```
@log_call              ← Step 1: Python sees the decorator
def add(a, b):         ← Step 2: Python creates the function object
    return a + b

# What Python actually does:
# 1. Creates the function object for add
# 2. Calls log_call(add)      ← decorator receives the function
# 3. Assigns the return value back to the name 'add'
#    add = log_call(add)       ← add now points to the wrapper
```

The decorator pattern involves **three functions**:

```
def log_call(func):           ← THE DECORATOR: receives the function
    @functools.wraps(func)
    def wrapper(*args, **kw):  ← THE WRAPPER: receives the function's arguments
        print(f"Calling {func.__name__}")   ← behavior BEFORE
        result = func(*args, **kw)           ← call the ORIGINAL function
        print(f"Returned {result}")          ← behavior AFTER
        return result                         ← return what the original returned
    return wrapper             ← return the wrapper (replaces original)
```

#### Decorator Factories — Three Levels of Nesting

When a decorator needs parameters (`@retry(max_attempts=3)`), you need an extra level:

```
def retry(max_attempts=3):     ← FACTORY: receives decorator's parameters
    def decorator(func):        ← DECORATOR: receives the function
        @functools.wraps(func)
        def wrapper(*args, **kw):  ← WRAPPER: receives function's arguments
            for attempt in range(max_attempts):  ← uses factory's parameter
                try:
                    return func(*args, **kw)
                except TransientError:
                    if attempt == max_attempts - 1:
                        raise
                    time.sleep(2 ** attempt)
        return wrapper
    return decorator            ← factory returns the decorator

# Execution:
# @retry(max_attempts=3)  →  retry(max_attempts=3) returns decorator
# def fetch(url): ...     →  decorator(fetch) returns wrapper
#                         →  fetch = wrapper
```

**Closure chain:** `wrapper` closes over `func` and `max_attempts`. `decorator` closes over `max_attempts`. Each level captures variables from the level above.

#### Object Lifecycle

```
             DEFINITION TIME                    CALL TIME
             ─────────────────                  ──────────
1. @retry(max_attempts=3)       → factory called, returns decorator
2. decorator(fetch_page)        → decorator called, returns wrapper
3. fetch_page = wrapper         → name rebound to wrapper

... later ...

4. fetch_page("http://...")     → wrapper called with url
5.   wrapper calls func(url)    → original fetch_page executes
6.   if exception → retry loop  → wrapper's control flow handles retry
```

#### Key Gotchas

| Gotcha | Cause | Fix |
|---|---|---|
| `decorated.__name__` returns `'wrapper'` | Wrapper replaces the original | Use `@functools.wraps(func)` |
| Late binding in closures over loop variables | All closures share the same variable cell | Use default argument: `def f(x=x):` |
| `*args, **kwargs` required in wrapper | Wrapper must accept any arguments the original does | Always use `wrapper(*args, **kwargs)` |
| Factory vs. plain decorator confusion | `@retry` vs `@retry()` — one is missing the factory call | Decide one interface and document it |

### 4. When to Use / When NOT to Use

**Use decorators when:**
- The behavior is a **cross-cutting concern** — it applies to many functions regardless of their specific purpose (retry, logging, timing, caching, authentication)
- The injected behavior doesn't need to know about the function's internals — it operates on the function as a black box (call it, handle its output/exceptions)
- You want to apply the behavior **declaratively** — `@retry` reads as "this function should be retried" without cluttering the body

**Do NOT use decorators when:**
- The behavior is **specific to one function** — just put it in the function body
- The wrapper needs deep knowledge of the function's parameters or return structure — use a pipeline/middleware pattern instead
- You need runtime-conditional behavior that changes per call — use dependency injection or strategy pattern
- The decorator stack gets too deep (4+ decorators) — debugging becomes painful because the call stack is full of wrappers

**Alternative for cross-cutting concerns:** middleware/pipeline pattern (Week 7). Better when the concern needs to inspect or transform the data flowing through, not just wrap execution.

### 5. ScraperFlow Connection

**File:** `scraperflow/retry.py`

**Component:** `@retry` decorator factory

The `@retry` decorator wraps `fetch_page()` in `scraperflow/fetcher.py`. The decorator:
- Catches only `TransientFetchError` (defined in `scraperflow/exceptions.py`)
- Applies exponential backoff with full jitter between attempts
- Re-raises after `max_attempts` exhausted
- Is completely transparent when the fetch succeeds on the first attempt

**Why a decorator is the right fit here:** retry logic is a cross-cutting concern — it doesn't care *what* is being fetched or *which site* the config points to. It only cares about the failure type. A decorator lets `fetch_page` stay clean (just fetch) while the retry policy lives separately, tested independently, applied declaratively.

```python
# scraperflow/fetcher.py
@retry(max_attempts=3, base_delay=1.0)
def fetch_page(url: str) -> str:
    """Fetch a URL. Raises TransientFetchError or PermanentFetchError."""
    # Just the core responsibility — no retry logic here
    ...
```

### 6. Worked Example — Closures

In [1]:
# Closure basics: a function that remembers its enclosing scope

def make_greeter(greeting):
    """Factory that returns a greeter function with a captured greeting."""
    def greet(name):
        # 'greeting' is a free variable — captured from make_greeter's scope
        return f"{greeting}, {name}!"
    return greet

hello = make_greeter("Hello")
howdy = make_greeter("Howdy")

print(hello("Alice"))  # Hello, Alice!
print(howdy("Bob"))    # Howdy, Bob!

# Inspect the closure:
print(f"Free variables: {hello.__code__.co_freevars}")  # ('greeting',)
print(f"Closure cells: {hello.__closure__}")             # (<cell ...>,)
print(f"Captured value: {hello.__closure__[0].cell_contents}")  # Hello

Hello, Alice!
Howdy, Bob!
Free variables: ('greeting',)
Closure cells: (<cell at 0x1044c45b0: str object at 0x104b99bf0>,)
Captured value: Hello


### Worked Example — Basic Decorator

In [2]:
import functools

def log_call(func):
    """Decorator that logs function calls and return values."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"→ Calling {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"← {func.__name__} returned {result}")
        return result
    return wrapper

@log_call
def add(a, b):
    """Add two numbers."""
    return a + b

result = add(3, 4)
print(f"Result: {result}")          # 7
print(f"Name: {add.__name__}")      # 'add' (preserved by @wraps)
print(f"Doc: {add.__doc__}")        # 'Add two numbers.' (preserved)
print(f"Wrapped: {add.__wrapped__}")  # original function reference

→ Calling add((3, 4), {})
← add returned 7
Result: 7
Name: add
Doc: Add two numbers.
Wrapped: <function add at 0x104ba18a0>


### Worked Example — Decorator Factory (Three Levels)

In [1]:
import functools
import time
import random


def retry(max_attempts=3, base_delay=1.0):
    """Decorator factory: retry on ValueError with exponential backoff + jitter."""
    # Level 1: Factory — captures max_attempts, base_delay
    
    def decorator(func):
        # Level 2: Decorator — captures func
        
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # Level 3: Wrapper — executes on each call
            last_exception = None
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except ValueError as exc:
                    last_exception = exc
                    if attempt < max_attempts - 1:
                        delay = random.uniform(0, base_delay * (2 ** attempt))
                        print(f"  Attempt {attempt + 1} failed: {exc}. "
                              f"Retrying in {delay:.2f}s...")
                        time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator


# Usage:
call_count = 0

@retry(max_attempts=3, base_delay=0.01)  # tiny delay for demo
def flaky_function():
    """Fails twice, then succeeds."""
    global call_count
    call_count += 1
    if call_count < 3:
        raise ValueError(f"Attempt {call_count} failed")
    return "success!"

result = flaky_function()
print(f"Result: {result}")
print(f"Total calls: {call_count}")

  Attempt 1 failed: Attempt 1 failed. Retrying in 0.01s...
  Attempt 2 failed: Attempt 2 failed. Retrying in 0.01s...
Result: success!
Total calls: 3


---
## CORE: Custom Exception Hierarchies

### 1. Terminology

| Term | Definition |
|---|---|
| **Exception hierarchy** | A tree of exception classes connected by inheritance. `except ParentError` catches all children. |
| **Base exception** | The root of your custom hierarchy — exists to give a single catch point for "any error from this system." |
| **Narrow catch** | Catching a specific, leaf-level exception. Preferred when different failures need different handling. |
| **Broad catch** | Catching a base exception. Useful for top-level error boundaries, dangerous inside business logic. |
| **Exception chaining** | Using `raise NewError() from original_error` to preserve the causal chain. Python stores it in `__cause__`. |
| **Failure classification** | Encoding *what kind* of failure occurred into the exception type, so handlers can act on type rather than inspecting error messages. |

### 2. What, Why & Consequences

**What is it?**

A custom exception hierarchy is a tree of exception classes that encodes your system's failure taxonomy into Python's type system. Instead of raising generic `Exception("timeout")` and string-matching in handlers, you raise `TransientFetchError` and catch by type.

**Why does it exist?**

Different failures demand different responses:
- **Transient** (timeout, 503) → retry after backoff
- **Permanent** (404, malformed URL) → give up immediately, log, move on
- **Bug** (TypeError, AttributeError) → crash loudly, fix the code

Without a hierarchy, you end up with:
```python
except Exception as e:
    if "timeout" in str(e) or "503" in str(e):  # Fragile string matching
        retry()
    elif "404" in str(e):
        skip()
```

With a hierarchy:
```python
except TransientFetchError:  # Type-safe, refactoring-safe, IDE-navigable
    retry()
except PermanentFetchError:
    skip()
```

**What goes wrong without it?**

- Retry logic catches too broadly and retries permanent failures (wasting time, hammering dead URLs)
- Retry logic catches too broadly and retries bugs (hiding `TypeError` from a code error)
- Error handling depends on fragile string matching that breaks when library error messages change
- No way to catch "all ScraperFlow errors" without also catching unrelated exceptions

### 3. How It Works (Internals)

#### Exception Resolution — How `except` Matches

Python's `except` uses `isinstance()` under the hood. When you write:

```python
except TransientFetchError:
```

Python checks: `isinstance(raised_exception, TransientFetchError)`. Because `isinstance` respects inheritance, catching a parent catches all children.

#### Hierarchy Design Pattern

```
ScraperFlowError (base — catches anything from our system)
├── FetchError (any failure in the fetch layer)
│   ├── TransientFetchError (worth retrying)
│   │   ├── TimeoutError_ (connection or read timeout)
│   │   ├── ServerError (5xx responses)
│   │   └── RateLimitError (429 — transient, back off longer)
│   └── PermanentFetchError (don't retry)
│       ├── NotFoundError (404)
│       └── ForbiddenError (403 — might be permanent or config issue)
└── StorageError (V6 — failures writing output)
```

#### Catch Semantics

```python
try:
    fetch_page(url)
except TransientFetchError:    # Catches TimeoutError_, ServerError, RateLimitError
    retry()                     # Only failures worth retrying
except PermanentFetchError:    # Catches NotFoundError, ForbiddenError
    skip()                      # Failures that won't resolve with retry
except ScraperFlowError:       # Catches everything above + future errors
    log_and_continue()          # Top-level error boundary
# TypeError, ValueError etc. propagate — they're bugs, not expected failures
```

#### Exception Chaining (`from`)

When translating a library exception into your hierarchy:

```python
try:
    response = requests.get(url, timeout=10)
except requests.Timeout as exc:
    raise TransientFetchError(f"Timeout fetching {url}") from exc
    # exc is stored in TransientFetchError.__cause__
    # Traceback shows both: "The above exception was the direct cause of..."
```

#### Key Design Rules

| Rule | Reason |
|---|---|
| Keep hierarchies shallow (2-3 levels) | Deep trees are hard to reason about; you rarely need more than base → category → specific |
| Leaf exceptions can be empty (`pass`) | Their type IS their information — the class name tells handlers what to do |
| Add attributes for context, not subclasses | `TransientFetchError(url=url, status=503)` rather than `HTTP503Error` |
| Always chain with `from` when translating | Preserves the original traceback for debugging |

### 4. When to Use / When NOT to Use

**Use custom exception hierarchies when:**
- Different failure categories require different handling strategies (retry vs. skip vs. crash)
- Multiple callers need to catch at different granularity levels (the decorator catches `TransientFetchError`, the CLI catches `ScraperFlowError`)
- You're wrapping a third-party library and want to translate its exceptions into your domain vocabulary

**Do NOT use when:**
- A single exception type is sufficient (no branching on failure type needed)
- The error represents a programming mistake — let it be a `TypeError`/`ValueError` and crash
- You're building leaf exceptions for every possible HTTP status code — over-engineering. Group by handling strategy, not by cause

**Alternative:** `errno`-style numeric codes in one exception class. Worse: no type-based dispatch, requires `if exc.code == ...` in handlers.

### 5. ScraperFlow Connection

**File:** `scraperflow/exceptions.py`

The hierarchy lives in its own module because it's **shared vocabulary** — imported by `fetcher.py` (to raise), `retry.py` (to catch), and in later versions by `worker.py` (to decide requeue vs. permanent failure) and `storage.py` (to add `StorageError`).

```python
# scraperflow/exceptions.py
class ScraperFlowError(Exception): ...
class FetchError(ScraperFlowError): ...
class TransientFetchError(FetchError): ...
class PermanentFetchError(FetchError): ...
```

The `@retry` decorator catches `TransientFetchError`. The fetcher raises `TransientFetchError` for 5xx/timeout and `PermanentFetchError` for 4xx. The hierarchy is the **contract** between these two components.

### 6. Worked Example — Custom Exception Hierarchy

In [2]:
class ScraperFlowError(Exception):
    """Base exception for all ScraperFlow errors."""

class FetchError(ScraperFlowError):
    """Base for all fetch-layer failures."""
    def __init__(self, url: str, message: str):
        self.url = url
        super().__init__(f"{message} (url={url})")

class TransientFetchError(FetchError):
    """Failure that may resolve on retry (timeout, 5xx, rate limit)."""

class PermanentFetchError(FetchError):
    """Failure that will NOT resolve on retry (404, 410, bad URL)."""


# Demonstrate catch semantics:
def simulate_fetch(url: str, status: int) -> str:
    if 500 <= status < 600:
        raise TransientFetchError(url, f"Server error {status}")
    if status == 404:
        raise PermanentFetchError(url, "Not found")
    return f"<html>content from {url}</html>"

# Catching at different levels:
for url, status in [("http://a.com", 503), ("http://b.com", 404), ("http://c.com", 200)]:
    try:
        result = simulate_fetch(url, status)
        print(f"✓ {url}: got content")
    except TransientFetchError as exc:
        print(f"⟳ {url}: transient — would retry ({exc})")
    except PermanentFetchError as exc:
        print(f"✗ {url}: permanent — skip ({exc})")

⟳ http://a.com: transient — would retry (Server error 503 (url=http://a.com))
✗ http://b.com: permanent — skip (Not found (url=http://b.com))
✓ http://c.com: got content


---
## SUPPORTING: `functools.wraps`

### 1. What & Why

`functools.wraps` is a decorator applied to the wrapper function inside a decorator. It copies metadata (`__name__`, `__doc__`, `__module__`, `__qualname__`, `__dict__`) from the original function to the wrapper, and sets `__wrapped__` to point at the original. Without it, every decorated function appears to be named `wrapper` in tracebacks, docs, and `help()`.

### 2. Key Usage Patterns

```python
import functools

def my_decorator(func):
    @functools.wraps(func)  # ← always add this line
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

| Attribute | Without `@wraps` | With `@wraps` |
|---|---|---|
| `func.__name__` | `'wrapper'` | `'fetch_page'` |
| `func.__doc__` | wrapper's docstring (or None) | original docstring |
| `func.__wrapped__` | doesn't exist | points to original function |
| Traceback | shows `wrapper` | shows original name |

### 3. ScraperFlow Connection

The `@retry` decorator in `scraperflow/retry.py` must use `@functools.wraps(func)` so that `fetch_page` retains its name in log messages and tracebacks. Without it, every retry log would say "Retrying wrapper" instead of "Retrying fetch_page."

---
## Architecture Concepts

**Core (full depth):** Cross-cutting concerns, exponential backoff with jitter

**Supporting (lighter):** Transient vs. permanent failure classification

---
## CORE: Cross-Cutting Concerns

### 1. Terminology

| Term | Definition |
|---|---|
| **Cross-cutting concern** | Functionality that applies to many components but isn't part of any one component's core responsibility. Examples: logging, retry, authentication, timing, caching. |
| **Core concern** | The primary responsibility of a component — the reason it exists. For `fetch_page`, the core concern is "make an HTTP request and return the body." |
| **Aspect** | In aspect-oriented programming (AOP), a modular unit of a cross-cutting concern. Python decorators serve a similar role. |
| **Separation of concerns** | The principle that each module should address one concern. Cross-cutting concerns challenge this — they touch everything, so they need a mechanism to be applied without being embedded. |

### 2. What, Why & Consequences

**What is it?**

A cross-cutting concern is functionality that:
1. Applies **uniformly** across many otherwise-unrelated components
2. Is **not** part of any one component's core responsibility
3. Would be **duplicated** if embedded directly in each component

The handbook (Chapter 2.4) frames it precisely:

> The first question isn't "how do I implement retry" — it's "what kind of thing *is* retry logic, conceptually?" Retry logic isn't about *what* is being fetched — it's about *how failures of a certain class should be handled*, which is a property of the failure, not the content.

**Why does it matter for ScraperFlow?**

```
┌─────────────────────────────────────────────────────┐
│                    Cross-cutting                      │
│              ┌────────────────────┐                   │
│              │   Retry Policy     │                   │
│              └────────┬───────────┘                   │
│                       │ wraps                         │
│    ┌──────────────────┼──────────────────┐           │
│    │                  │                  │           │
│    ▼                  ▼                  ▼           │
│ fetch_page()   fetch_sitemap()   fetch_robots()     │
│ (core: HTTP)   (core: HTTP)      (core: HTTP)       │
│                                                      │
│ Each function's CORE concern is different.           │
│ The RETRY concern is identical across all.           │
└─────────────────────────────────────────────────────┘
```

**Consequences of embedding cross-cutting concerns:**
- **Tangling:** core logic and cross-cutting logic mixed in one function — harder to read, harder to test each independently
- **Scattering:** the same cross-cutting logic repeated in many functions — change in one place requires change in all
- **Inconsistency:** each copy drifts slightly — different retry counts, different backoff formulas, different exception handling

**The solution:** extract the cross-cutting concern into one place (a decorator) and apply it declaratively.

### 3. How It Works (Internals)

#### The Decision Framework

Ask: "Is this behavior about *what* my function does, or about *how to handle* a class of situations that could happen to any function?"

- If it's about *what* → core concern → put it in the function body
- If it's about *how to handle* → cross-cutting → extract into a decorator/middleware

#### Common Cross-Cutting Concerns in Web Scraping Systems

| Concern | Mechanism | ScraperFlow Version |
|---|---|---|
| Retry/backoff | Decorator | V3 (this week) |
| Logging | stdlib `logging` | V1 (already done) |
| Timing/metrics | Decorator or middleware | V8 |
| Rate limiting | Scheduler/middleware | V7 |
| Authentication | Middleware/header injection | (not in roadmap) |
| Caching | Decorator | (not in roadmap) |

#### Why Decorators Are the Right Tool (for V3)

Retry operates at the **function boundary** — it wraps the entire call, doesn't inspect arguments or transform data. It needs:
1. To call the function
2. To catch a specific exception type
3. To decide whether to call again or give up

This is exactly the pattern a decorator provides: wrap → try → decide → possibly repeat.

The *alternative* would be inline retry, which tangles the cross-cutting concern with the core logic and makes both harder to test independently.

### 4. ScraperFlow Connection

The `@retry` decorator in `scraperflow/retry.py` is ScraperFlow's first explicit cross-cutting concern. It's applied to `fetch_page` but has no knowledge of what's being fetched, what site is configured, or what happens with the result. It only knows about failure classification — "is this exception transient?" — which is exactly what makes it cross-cutting.

---
## CORE: Exponential Backoff with Jitter

### 1. Terminology

| Term | Definition |
|---|---|
| **Exponential backoff** | Waiting longer between each retry: delay = base × 2^attempt. Attempt 0: 1s, attempt 1: 2s, attempt 2: 4s, attempt 3: 8s. |
| **Jitter** | Adding randomness to the backoff delay to prevent synchronized retries from multiple clients. |
| **Full jitter** | `delay = random.uniform(0, min(cap, base × 2^attempt))` — the delay is random between 0 and the exponential ceiling. |
| **Equal jitter** | `delay = half + random.uniform(0, half)` where `half = min(cap, base × 2^attempt) / 2` — half fixed, half random. |
| **Decorrelated jitter** | `delay = random.uniform(base, previous_delay × 3)` — each delay depends on the previous one, not the attempt number. |
| **Thundering herd** | When many clients retry simultaneously (because they all use the same deterministic backoff), overwhelming the server at predictable intervals. |
| **Cap / max_delay** | Upper bound on delay to prevent absurdly long waits (e.g., 2^10 = 1024 seconds without a cap). |

### 2. What, Why & Consequences

**What is it?**

Exponential backoff with jitter is a retry strategy that:
1. Waits longer between each successive retry (exponential growth)
2. Adds randomness to that wait (jitter)
3. Caps the maximum wait to prevent absurd delays

**Why exponential (not linear or fixed)?**

If a server is struggling under load:
- **Fixed delay** (retry every 1s): adds constant load — if 100 clients are retrying every second, that's 100 req/s of retry traffic forever
- **Linear delay** (1s, 2s, 3s...): better, but still aggressive — retries thin out slowly
- **Exponential** (1s, 2s, 4s, 8s...): retries thin out quickly — gives the server breathing room to recover

**Why jitter (not deterministic exponential)?**

Without jitter, 100 clients that all failed at the same time will all retry at exactly t=1s, then exactly t=3s (1+2), then exactly t=7s (1+2+4)... They stay synchronized forever — a repeating thundering herd.

With full jitter, those 100 clients spread their retries randomly across the exponential window — the server sees a smooth trickle of retries instead of synchronized spikes.

```
Without jitter:          With full jitter:

Load                     Load
▲ │ ██                   ▲ │
  │ ██   ██              │ ▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  │ ██   ██   ██        │ ▒▒▒▒▒▒▒▒▒▒▒▒▒▒
  └──────────────→ t      └──────────────────→ t
  Spikes at 1s,3s,7s     Spread evenly over time
```

**What goes wrong without jitter?**

AWS's analysis ("Exponential Backoff and Jitter") showed that without jitter, competing clients complete their work **significantly** slower than with jitter, because the synchronized spikes keep overwhelming the server.

### 3. How It Works (Internals)

#### The Full Jitter Formula

```python
import random

def calculate_delay(attempt: int, base_delay: float = 1.0, max_delay: float = 60.0) -> float:
    """Full jitter: random between 0 and the exponential ceiling."""
    exponential_ceiling = min(max_delay, base_delay * (2 ** attempt))
    return random.uniform(0, exponential_ceiling)
```

#### Attempt-by-Attempt Breakdown

With `base_delay=1.0`, `max_delay=60.0`:

| Attempt | Exponential ceiling | Jitter range | Example delay |
|---|---|---|---|
| 0 | min(60, 1×2⁰) = 1.0 | [0, 1.0] | 0.73s |
| 1 | min(60, 1×2¹) = 2.0 | [0, 2.0] | 1.42s |
| 2 | min(60, 1×2²) = 4.0 | [0, 4.0] | 2.91s |
| 3 | min(60, 1×2³) = 8.0 | [0, 8.0] | 5.17s |
| 4 | min(60, 1×2⁴) = 16.0 | [0, 16.0] | 9.83s |
| 5 | min(60, 1×2⁵) = 32.0 | [0, 32.0] | 18.44s |
| 6 | min(60, 1×2⁶) = 60.0 | [0, 60.0] | 42.17s |
| 7+ | min(60, 1×2⁷) = 60.0 | [0, 60.0] | (capped) |

#### Why Full Jitter Wins

AWS tested three strategies with competing clients:

| Strategy | Formula | Completion time (relative) |
|---|---|---|
| No jitter | `base × 2^attempt` | Slowest (thundering herd) |
| Equal jitter | `half + random(0, half)` | Better |
| **Full jitter** | `random(0, base × 2^attempt)` | **Best** |
| Decorrelated | `random(base, prev × 3)` | Comparable to full |

Full jitter wins because it maximizes spread — some clients retry almost immediately (getting lucky), others wait nearly the full window. This smooths out the load curve.

### 4. ScraperFlow Connection

The `@retry` decorator in `scraperflow/retry.py` uses full jitter between attempts:

```python
delay = random.uniform(0, min(max_delay, base_delay * (2 ** attempt)))
time.sleep(delay)
```

This matters even for a single-worker scraper: multiple URLs may hit the same struggling server, and jitter prevents synchronized retry storms during batch processing.

### 5. Worked Example — Exponential Backoff with Jitter

In [ ]:
import random

def calculate_delay(attempt: int, base_delay: float = 1.0, max_delay: float = 60.0) -> float:
    """Full jitter backoff: random between 0 and exponential ceiling."""
    ceiling = min(max_delay, base_delay * (2 ** attempt))
    return random.uniform(0, ceiling)

# Show the exponential growth + jitter spread
print("Attempt | Ceiling | Sample delays (3 trials)")
print("-" * 55)
for attempt in range(7):
    ceiling = min(60.0, 1.0 * (2 ** attempt))
    samples = [calculate_delay(attempt) for _ in range(3)]
    samples_str = ", ".join(f"{s:.2f}s" for s in samples)
    print(f"   {attempt}    | {ceiling:5.1f}s  | {samples_str}")

print("\n--- Thundering herd comparison ---")
print("10 clients, attempt 3, NO jitter:  all wait exactly", 1.0 * 2**3, "s")
print("10 clients, attempt 3, full jitter:", 
      [f"{calculate_delay(3):.2f}s" for _ in range(10)])

---
## SUPPORTING: Transient vs. Permanent Failure Classification

### 1. What & Why

Not all failures are equal. A **transient** failure is one that may resolve if you wait and try again — the server was momentarily overloaded, the network had a brief hiccup. A **permanent** failure won't resolve no matter how many times you retry — the page doesn't exist, the URL is malformed, authentication is rejected.

The classification determines what the retry decorator should do: retry transient failures (they might work next time), propagate permanent failures immediately (don't waste time).

### 2. Key Classification Rules

| Signal | Classification | Reasoning |
|---|---|---|
| HTTP 5xx (500, 502, 503, 504) | Transient | Server-side issue, likely temporary |
| HTTP 429 (Too Many Requests) | Transient | Rate limited — back off and retry |
| Connection timeout | Transient | Network congestion, server busy |
| Connection refused | Transient | Server might be restarting |
| DNS resolution failure | Transient (usually) | DNS can be flaky |
| HTTP 404 (Not Found) | Permanent | Page doesn't exist |
| HTTP 410 (Gone) | Permanent | Explicitly removed |
| HTTP 400 (Bad Request) | Permanent | Our request is malformed |
| HTTP 401/403 (Auth) | Permanent* | Credentials won't magically appear |
| Malformed URL | Permanent | The input is wrong |

\* 403 is debatable — could be transient if it's rate-limit-based, permanent if it's actual authorization. Default to permanent; override per-site if needed.

### 3. ScraperFlow Connection

In `scraperflow/fetcher.py`, the fetch function translates raw `requests` exceptions and HTTP status codes into `TransientFetchError` or `PermanentFetchError`. This classification is the **contract** between the fetcher (which knows about HTTP) and the retry decorator (which only knows about exception types). The decorator never inspects status codes — it trusts the hierarchy.

---
## Testing Concepts

## SUPPORTING: Monkeypatching `time.sleep`

### 1. What & Why

Testing retry logic with real `time.sleep` means your test suite takes *seconds to minutes* for what should be millisecond tests. Monkeypatching replaces `time.sleep` with a fake that records the requested delay without actually waiting.

### 2. Key Usage Patterns

**With pytest's `monkeypatch` fixture:**
```python
def test_retry_backoff(monkeypatch):
    sleep_calls = []
    monkeypatch.setattr("time.sleep", lambda seconds: sleep_calls.append(seconds))
    
    # Exercise the retry decorator...
    
    assert len(sleep_calls) == 2  # retried twice
    assert sleep_calls[0] < sleep_calls[1]  # backoff increased
```

**With `unittest.mock.patch`:**
```python
from unittest.mock import patch

def test_retry_backoff():
    with patch("scraperflow.retry.time.sleep") as mock_sleep:
        # Exercise the retry decorator...
        
        assert mock_sleep.call_count == 2
        delays = [call.args[0] for call in mock_sleep.call_args_list]
        assert delays[0] < delays[1]
```

**Important:** Patch `time.sleep` where it's *used* (in your retry module), not where it's *defined* (in the `time` module). If `retry.py` does `import time` then patch `scraperflow.retry.time.sleep`.

### 3. ScraperFlow Connection

Every test in `tests/test_retry.py` monkeypatches `time.sleep` so the full retry test suite runs in milliseconds. The tests verify:
- Correct number of sleep calls (= number of retries)
- Increasing delay pattern (exponential backoff)
- Non-deterministic delays (jitter — delays differ across runs)

The "Done when" criterion for this week: *the suite still runs in well under a second despite testing multi-attempt logic.*

---
## Best Practices

### Python — Decorators
- **Always use `@functools.wraps(func)`** — every decorator should preserve the wrapped function's metadata. This is non-negotiable for debugging.
- **Accept `*args, **kwargs` in the wrapper** — don't restrict the decorated function's signature unless you intentionally want to.
- **Keep decorators focused** — one decorator, one concern. Don't build a decorator that does retry AND logging AND timing.
- **Document what exception types the decorator catches** — callers need to know what gets swallowed vs. propagated.
- **Test decorators independently** — a decorated function conflates two things. Test the decorator's behavior with a trivial stub function.

### Python — Exceptions
- **Inherit from a project base exception** — gives callers a single "catch everything from this library" option.
- **Chain exceptions with `from`** — always `raise MyError() from original` to preserve the diagnostic trail.
- **Add attributes, not subclasses, for context** — `FetchError(url=url, status=503)` not `HTTP503Error`.
- **Never catch `Exception` inside a retry loop** — you'll hide bugs (`TypeError`, `AttributeError`) that should crash immediately.

### Architecture — Retry
- **Always have a maximum attempt count** — infinite retries are a production incident.
- **Always include jitter** — deterministic backoff creates thundering herds.
- **Cap the maximum delay** — without a cap, attempt 10 waits 1024 seconds.
- **Log every retry** — include attempt number, delay, and the error that triggered it.
- **Re-raise after exhaustion** — don't swallow the failure. The caller must know.

### Testing
- **Monkeypatch blocking calls** — `time.sleep`, `requests.get`, anything that makes tests slow.
- **Test the retry decorator with a controllable stub** — a function that fails N times then succeeds.
- **Verify both success and exhaustion paths** — the decorator must work correctly in both.
- **Don't test jitter values exactly** — test that delays are within expected ranges, not exact values.

---
## Common Mistakes

### Python — Decorator Mistakes

1. **Forgetting `@functools.wraps(func)`** — decorated functions lose their name, docstring, and become impossible to debug in tracebacks.

2. **Confusing decorator and decorator factory:** `@retry` vs `@retry()`. If `retry` is a factory, writing `@retry` passes the function as the `max_attempts` parameter — a confusing `TypeError` that's hard to diagnose.

3. **Not returning the function's result from the wrapper:**
   ```python
   def wrapper(*args, **kwargs):
       func(*args, **kwargs)  # Missing 'return'!
   ```
   The decorated function silently returns `None` instead of its actual value.

4. **Hardcoding the exception type to catch:**
   ```python
   except Exception:  # Too broad! Catches bugs too.
       retry()
   ```
   Only catch the specific exception type that represents a retriable failure.

5. **Late binding closure bug:**
   ```python
   decorators = []
   for i in range(3):
       def dec(func):
           def wrapper():
               print(i)  # Always prints 2! (last value of i)
               return func()
           return wrapper
       decorators.append(dec)
   ```
   Fix: use a default argument `def dec(func, i=i):` to capture the current value.

### Architecture — Retry Mistakes

1. **Retrying permanent failures** — a 404 will still be a 404 on attempt 5. Classify first, retry second.

2. **No backoff (immediate retry)** — hammers an already-struggling server. Makes the problem worse.

3. **Deterministic backoff without jitter** — creates synchronized retry storms when multiple workers hit the same server.

4. **No maximum attempt count** — infinite retry loop that never gives up = hung process.

5. **Swallowing the exception after exhaustion** — returning `None` instead of re-raising. The caller doesn't know the operation failed.

6. **Catching too broadly then classifying inside the handler:**
   ```python
   except Exception as e:
       if is_transient(e):  # Fragile — what about TypeError?
           retry()
   ```
   Better: catch narrowly with a typed hierarchy.

---
## Comparisons

### Decorator vs. Other Approaches for Cross-Cutting Concerns

| Approach | Mechanism | Pros | Cons | Best for |
|---|---|---|---|---|
| **Decorator** | Wraps function call | Declarative (`@retry`), testable in isolation, composable | Hidden control flow, can't inspect function args easily | Retry, caching, timing, auth |
| **Inline code** | Copy-pasted in each function | Explicit, no magic | DRY violation, inconsistent, untestable | Never (for cross-cutting) |
| **Base class method** | Inherited by all subclasses | One definition | Tight coupling, requires inheritance hierarchy, hard to apply selectively | Rarely appropriate |
| **Middleware/Pipeline** | Stage in a processing chain | Can inspect/transform data | More complex setup, overkill for simple wrapping | Data transformation, request processing (Week 7) |
| **Context manager** | `with` block wraps a section | Explicit scope, cleanup guaranteed | Can't wrap a whole function declaratively | Resource cleanup, transactions |

### Exception Hierarchy vs. Alternatives

| Approach | How failure type is encoded | Dispatch mechanism | Pros | Cons |
|---|---|---|---|---|
| **Exception hierarchy** | In the type (class) | `except SpecificError:` | Type-safe, IDE-navigable, catch at any level | Requires upfront design |
| **Error codes in one exception** | Numeric/string attribute | `if exc.code == 503:` | Simple to add new codes | No type-based dispatch, verbose handlers |
| **String matching** | In the error message | `if "timeout" in str(exc):` | Zero upfront design | Fragile, breaks on message changes |
| **Result types (Ok/Err)** | In return value | Pattern matching / `if` | No exceptions, explicit flow | Verbose, not Pythonic for I/O errors |

### Backoff Strategies

| Strategy | Formula | Spread | Thundering herd risk | Complexity |
|---|---|---|---|---|
| Fixed delay | `delay = constant` | None | High | Trivial |
| Linear backoff | `delay = base × attempt` | None | High | Trivial |
| Exponential (no jitter) | `delay = base × 2^attempt` | None | High (synchronized spikes) | Low |
| **Exponential + full jitter** | `random(0, base × 2^attempt)` | Maximum | **Lowest** | Low |
| Exponential + equal jitter | `half + random(0, half)` | Medium | Low | Low |
| Decorrelated jitter | `random(base, prev × 3)` | High | Low | Medium |

---
## Real-World Use Cases

| Domain | How these concepts appear |
|---|---|
| **Web Scraping** | Scrapy's `RetryMiddleware` — retries on configurable HTTP codes with exponential backoff. ScraperFlow V3: `@retry` decorator on `fetch_page`. |
| **Backend Development** | API clients (`boto3`, `google-cloud-*`) all implement exponential backoff with jitter for transient AWS/GCP failures. Django's `@login_required` is a cross-cutting auth decorator. |
| **Data Engineering** | Airflow task retries with exponential backoff. dbt retry on warehouse connection failures. Every ETL pipeline needs transient failure handling. |
| **AI/ML** | Training pipelines retry on GPU OOM (transient if batch size can be reduced) vs. data corruption (permanent). MLflow experiment tracking uses decorators for metric logging. |
| **APIs** | Rate-limited APIs (GitHub, Twitter, Stripe) return 429 — clients must implement backoff with jitter. `requests` + `urllib3.Retry` is the standard pattern. |
| **Distributed Systems** | gRPC interceptors (decorator-like) handle retry. Kafka producers retry transient broker failures. Redis clients retry on connection reset. |
| **Testing** | pytest fixtures use decorator machinery. `@pytest.mark.parametrize` is a decorator factory. Mocking with `@patch` is a decorator that modifies test behavior. |

---
## Official References

### Python Documentation
- [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps) — metadata preservation for decorators
- [PEP 318 — Decorators for Functions and Methods](https://peps.python.org/pep-0318/) — the original decorator PEP
- [Python Exception Hierarchy](https://docs.python.org/3/library/exceptions.html#exception-hierarchy) — built-in exception tree
- [Defining Clean-up Actions (try/except/finally)](https://docs.python.org/3/tutorial/errors.html) — exception handling tutorial
- [`random.uniform`](https://docs.python.org/3/library/random.html#random.uniform) — for jitter calculation

### Architecture References
- [Exponential Backoff and Jitter — AWS Architecture Blog](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/) — the canonical analysis comparing jitter strategies
- [Retry Pattern — Microsoft Azure Architecture](https://learn.microsoft.com/en-us/azure/architecture/patterns/retry) — comprehensive retry guidance
- [Circuit Breaker Pattern — Martin Fowler](https://martinfowler.com/bliki/CircuitBreaker.html) — the next resilience pattern after retry (deferred to later versions)

### Library References
- [`tenacity` documentation](https://tenacity.readthedocs.io/) — production-grade retry library for Python
- [Scrapy RetryMiddleware source](https://github.com/scrapy/scrapy/blob/master/scrapy/downloadermiddlewares/retry.py) — mature implementation to study
- [`urllib3` Retry](https://urllib3.readthedocs.io/en/stable/reference/urllib3.util.html#urllib3.util.Retry) — transport-level retry

### Books
- *Fluent Python* (Ramalho) — Chapters 9-10: closures, decorators, and the mechanics underneath
- *Release It!* (Nygard) — stability patterns including Circuit Breaker, Timeouts, Bulkheads

---
## Practice Exercises

### Easy
1. Write a closure `make_counter()` that returns a function incrementing a captured count variable.
2. Write a basic decorator `log_call` that prints the function name before and after execution.
3. Create an exception hierarchy: `AppError` → `NetworkError` → `TimeoutError_`. Write a `try/except` that catches `NetworkError` and verify it catches `TimeoutError_` too.
4. Calculate the exponential backoff ceiling (without jitter) for attempts 0 through 6 with base_delay=1.0.

### Medium
5. Build a decorator factory `@repeat(n)` that calls the function `n` times and returns all results as a list.
6. Build a decorator `@suppress(ValueError, TypeError)` that catches specified exceptions and returns `None` instead of raising.
7. Design an exception hierarchy for a payment system: distinguish between retryable failures (gateway timeout), permanent failures (insufficient funds), and configuration errors (invalid API key). Justify your tree structure.
8. Implement `calculate_delay()` with full jitter. Generate 100 delays for attempt 3 and verify they're all in [0, 8.0] and not all the same value.
9. Write a test for a retry decorator that monkeypatches `time.sleep` and verifies the number of sleep calls equals the number of retries.

### Hard
10. Build a complete `@retry(max_attempts, base_delay, max_delay)` decorator factory with full jitter. Test it with monkeypatched sleep.
11. Refactor this function to separate cross-cutting concerns (retry, logging, timing) into independent decorators:
    ```python
    def fetch(url):
        start = time.time()
        for attempt in range(3):
            try:
                print(f"Fetching {url}")
                resp = requests.get(url)
                print(f"Done in {time.time()-start:.2f}s")
                return resp.text
            except Timeout:
                time.sleep(2**attempt)
        raise FetchError("gave up")
    ```
12. **Architecture reasoning:** Your retry decorator currently uses a fixed `max_attempts=3` for all functions. A teammate proposes making it per-site (some sites need 5 retries, others need 2). Design how you'd support this without changing the decorator's interface. Justify whether the complexity is worth it today vs. deferring.

---
## Interview Questions

### Beginner
1. What is a closure in Python? How does an inner function access variables from its enclosing scope?
2. What does `@functools.wraps(func)` do, and why should you use it in every decorator?
3. What is the difference between a decorator and a decorator factory?
4. Why do custom exceptions inherit from `Exception` (not `BaseException`)?
5. What is exponential backoff? Why use it instead of a fixed delay?

### Intermediate
6. Why use a decorator for retry logic instead of inline `try/except` with a loop at each call site? (Cross-cutting concern reasoning)
7. Why does backoff need jitter, specifically? What problem does jitter solve that plain exponential backoff doesn't?
8. How do you test retry logic without your test suite taking real minutes to run?
9. Explain the difference between transient and permanent failures. Give three examples of each in the context of web scraping.
10. What happens if your retry decorator catches `Exception` instead of a specific exception type? What bugs could this hide?

### Advanced
11. Compare full jitter, equal jitter, and decorrelated jitter. Under what workload characteristics does each perform best?
12. Your retry decorator gives up after 3 attempts. But in production, a domain is intermittently down for 30 minutes. The retry decorator can't help (its total backoff is ~7 seconds). Design a second resilience layer that handles this longer time horizon without changing the decorator. (Hint: requeue mechanism)
13. You're debugging a production issue where a decorated function's traceback shows `wrapper` instead of the real function name. What went wrong and how do you fix it? How do you prevent this systematically?
14. When would you use a third-party retry library (like `tenacity`) instead of a hand-rolled decorator? What trade-offs are you making?
15. Design a circuit breaker that works alongside your retry decorator. How do the two patterns complement each other? When should the circuit breaker "open" and prevent retries entirely?

---
## Summary

### Key Takeaways

- **Closures** are functions that capture variables from their enclosing scope. The captured values live in `__closure__` cell objects, surviving after the enclosing function returns.

- **Decorators** use closures to wrap behavior around functions without modifying their bodies. The pattern: decorator receives function → returns wrapper → wrapper adds behavior and calls original.

- **Decorator factories** add one more level: factory receives parameters → returns decorator → decorator receives function → returns wrapper. Three levels of nesting, each capturing from the level above.

- **`functools.wraps`** preserves the original function's metadata on the wrapper. Always use it.

- **Custom exception hierarchies** encode failure classification into the type system. `except TransientFetchError` is type-safe, IDE-navigable, and doesn't break when error messages change.

- **Cross-cutting concerns** are behaviors that apply uniformly across many components but aren't any one component's core responsibility. Decorators are the Python idiom for applying them declaratively.

- **Exponential backoff** gives struggling servers time to recover. **Jitter** prevents synchronized retries from creating thundering herds. **Full jitter** (`random(0, base × 2^attempt)`) provides maximum spread.

- **Monkeypatching `time.sleep`** keeps retry tests fast. Patch where it's used, not where it's defined.

### The ScraperFlow V3 Architecture

```
scraperflow/exceptions.py     ← Shared vocabulary (TransientFetchError, PermanentFetchError)
scraperflow/retry.py          ← @retry decorator (catches TransientFetchError, backs off with jitter)
scraperflow/fetcher.py        ← Raises typed exceptions, decorated with @retry
tests/test_retry.py           ← Tests decorator in isolation (monkeypatched sleep)
tests/test_fetcher.py         ← Tests exception classification
```

### One-Line Summary

Retry is a cross-cutting concern → lives in a decorator → catches only typed transient failures → backs off with exponential jitter → gives up loudly after a bounded number of attempts.